In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define your paths (Adjust these to point to your specific folder structures)
DATASET_DIR = "/content/drive/MyDrive/colon_cancer_dataset"
JSON_PATH = "/content/drive/MyDrive/colon_cancer_dataset/segmented-images/bounding-boxes.json"
IMAGES_DIR = "/content/drive/MyDrive/colon_cancer_dataset/segmented-images" # Corrected path

print(f"Drive successfully connected. Looking for configurations in: {DATASET_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive successfully connected. Looking for configurations in: /content/drive/MyDrive/colon_cancer_dataset


In [ ]:
import os
import json
import cv2
import shutil
from sklearn.model_selection import train_test_split
from google.colab import drive

# 1. Mount Google Drive safely
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Hardcoded Absolute Target Directory Mapping
# Adjust folder hierarchy paths based on your actual workspace directory structure
DRIVE_DATASET_DIR = '/content/drive/MyDrive/colon_cancer_dataset'
JSON_PATH = '/content/drive/MyDrive/colon_cancer_dataset/segmented-images/bounding-boxes.json'

# Output environment within Colab virtual machine local storage
LOCAL_OUTPUT_DIR = '/content/yolov8_dataset/'
TARGET_SIZE = (640, 640)

# Verify path structure initialization to intercept FileNotFoundError early
if not os.path.exists(JSON_PATH):
    raise FileNotFoundError(f"Missing label artifact! Ensure bounding-boxes.json is located at: {JSON_PATH}")

print("Successfully verified labels structure root.")

# 3. Read bounding-boxes metadata
with open(JSON_PATH, 'r') as f:
    data = json.load(f)

# Explicitly discover valid images relative to JSON keys inside the Drive folder
valid_records = []
supported_extensions = ['.jpg', '.png', '.jpeg']

# Dynamically locate the base path where images reside inside your folder
# Corrected path for images based on JSON_PATH location and its subdirectories
images_search_root = os.path.join(os.path.dirname(JSON_PATH), 'images')

print(f"Scanning for dataset match items inside: {images_search_root}")

# Debugging: List contents of the images_search_root directory
print(f"Contents of {images_search_root}:\n{os.listdir(images_search_root)}")

for img_id, meta in data.items():
    found = False
    for ext in supported_extensions:
        potential_path = os.path.join(images_search_root, f"{img_id}{ext}")
        if os.path.exists(potential_path):
            valid_records.append((img_id, potential_path, meta))
            found = True
            break

if len(valid_records) == 0:
    raise FileNotFoundError(f"Could not link image source references within: {images_search_root}. Check file extensions.")

print(f"Matched {len(valid_records)} image items for processing.")

# Define class indexes map based explicitly on dataset configuration schema
class_to_idx = {"polyp": 0}

# 4. Initialize Local Directory Ecosystem
subdirs = ['images/train', 'images/val', 'labels/train', 'labels/val']
for subdir in subdirs:
    os.makedirs(os.path.join(LOCAL_OUTPUT_DIR, subdir), exist_ok=True)

# 5. Stratified Data Splitting (Train/Validation Splits)
train_records, val_records = train_test_split(valid_records, test_size=0.2, random_state=42)

def convert_to_yolo_format(size, box):
    """Normalizes coordinate values into center relative offsets."""
    dw = 1.0 / size[0]
    dh = 1.0 / size[1]
    x_center = (box[0] + box[2]) / 2.0
    y_center = (box[1] + box[3]) / 2.0
    width = box[2] - box[0]
    height = box[3] - box[1]
    return (x_center * dw, y_center * dh, width * dw, height * dh)

def process_and_save(split_records, split_name):
    for img_id, src_path, meta in split_records:
        orig_w = meta['width']
        orig_h = meta['height']

        # Read image file safely
        img = cv2.imread(src_path)
        if img is None:
            continue

        # Standardize pixel layout scale to target dimensions
        resized_img = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_LINEAR)

        # Save preprocessed asset locally
        dest_img_name = f"{img_id}.jpg"
        cv2.imwrite(os.path.join(LOCAL_OUTPUT_DIR, 'images', split_name, dest_img_name), resized_img)

        # Generate companion TXT coordinate map tracking label tags
        label_file_path = os.path.join(LOCAL_OUTPUT_DIR, 'labels', split_name, f"{img_id}.txt")
        with open(label_file_path, 'w') as lf:
            for bbox in meta.get('bbox', []):
                cls_idx = class_to_idx.get(bbox['label'], 0)

                # Fetch absolute corner elements safely
                xmin, ymin = bbox['xmin'], bbox['ymin']
                xmax, ymax = bbox['xmax'], bbox['ymax']

                # Normalize mapping scale transformations
                yolo_box = convert_to_yolo_format((orig_w, orig_h), (xmin, ymin, xmax, ymax))
                lf.write(f"{cls_idx} {' '.join([f'{coord:.6f}' for coord in yolo_box])}\n")

# Execute conversion tasks across data frames partitions
process_and_save(train_records, 'train')
process_and_save(val_records, 'val')

# Write tracking dataset descriptor mapping context parameters
yaml_content = f"""
path: {LOCAL_OUTPUT_DIR}
train: images/train
val: images/val

names:
  0: polyp
"""

with open(os.path.join(LOCAL_OUTPUT_DIR, 'dataset.yaml'), 'w') as yf:
    yf.write(yaml_content.strip())

print(f"\nPreprocessing phase absolute success. Dataset locked at: {LOCAL_OUTPUT_DIR}")

Successfully verified labels structure root.
Scanning for dataset match items inside: /content/drive/MyDrive/colon_cancer_dataset/segmented-images/images
Contents of /content/drive/MyDrive/colon_cancer_dataset/segmented-images/images:
['050ffdb1-d81d-4c69-a5b9-751768997cd3.jpg', '02cac873-236f-40bd-a32a-811d6f8fac4c.jpg', '0198bc50-169e-456c-a56b-970a7f7d23b7.jpg', '01503109-d81f-404e-a919-31de1aceb6b9.jpg', '01dcc13a-85ab-4d6e-b42b-5d631e076ac2.jpg', '038d119f-85bf-4f42-897b-677eb1450b94.jpg', '0017b7c7-90f8-4de2-8723-1d87e5c58317.jpg', '035a608d-30f5-4fe8-8753-ce271b32b546.jpg', '00f98835-8fd8-43fe-960d-f4a1159c30f1.jpg', '01c95ee8-79cc-46f4-a6a0-834d8a6045d4.jpg', '0048d8c5-b59d-461c-9834-f44a727e191d.jpg', '01dbf397-c0fc-40e7-9b04-86de70781c9a.jpg', '0603cd1e-53bf-43d7-8a74-d424ff6a6bb2.jpg', '01e6cfd7-bdb6-4929-9fc0-9a7dd2d3c7f7.jpg', '0004a718-546c-41c2-9c69-c4685093a039.jpg', '026e9707-e74d-434f-98f4-6ceb2c707577.jpg', '08a43621-f3b8-493a-978d-0c802f1cad82.jpg', '14552594-a205-4

In [ ]:
import json
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
import shutil
import os

# Output YOLO directory scaffolding
OUTPUT_DIR = "/content/yolov8_dataset"
for split in ['train', 'val']:
    os.makedirs(os.path.join(OUTPUT_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, 'labels', split), exist_ok=True)

# Define target dimensions
TARGET_SIZE = 640
# Map text labels to integer classes
LABEL_MAP = {"polyp": 0}

# Assuming JSON_PATH is defined in a previous cell or globally available
# Define JSON_PATH and IMAGES_DIR explicitly for this cell's context
DRIVE_DATASET_DIR = '/content/drive/MyDrive/colon_cancer_dataset'
JSON_PATH = '/content/drive/MyDrive/colon_cancer_dataset/segmented-images/bounding-boxes.json'
IMAGES_DIR = os.path.join(os.path.dirname(JSON_PATH), 'images')

# Read your parsed file details
if not os.path.exists(JSON_PATH):
    print(f"ERROR: JSON file not found at {JSON_PATH}")
    print("Please ensure the 'bounding-boxes.json' file exists in your Google Drive folder defined by DATASET_DIR.")
else:
    with open(JSON_PATH, 'r') as f:
        annotations = json.load(f)

    # Collect all valid records to prepare a balanced distribution setup
    valid_records = []
    for img_name, meta in annotations.items():
        # Append common extensions if not implicitly declared in JSON keys
        img_filename = img_name if img_name.endswith('.jpg') else f"{img_name}.jpg"
        img_path = os.path.join(IMAGES_DIR, img_filename)

        if os.path.exists(img_path):
            valid_records.append((img_filename, meta))
        else:
            print(f"DEBUG: Image file not found at {img_path}") # Debugging line

    print(f"Found {len(valid_records)} matching files for transformation processing.")

    # Execute stratified/balanced distribution splits (e.g., 80% Train, 20% Val)
    # Only proceed if there are valid records to split
    if valid_records:
        train_data, val_data = train_test_split(valid_records, test_size=0.2, random_state=42)
    else:
        train_data, val_data = [], [] # Ensure these are defined as empty lists if no valid records
        print("No valid records found to split into training and validation sets.")

    def process_and_save(dataset_split, split_name):
        for filename, meta in dataset_split:
            img_path = os.path.join(IMAGES_DIR, filename)
            img = cv2.imread(img_path)
            if img is None:
                print(f"Warning: Could not read image {img_path}. Skipping.")
                continue
            orig_h, orig_w = meta['height'], meta['width']

            # 1. Image Resize Operation
            resized_img = cv2.resize(img, (TARGET_SIZE, TARGET_SIZE))
            cv2.imwrite(os.path.join(OUTPUT_DIR, 'images', split_name, filename), resized_img)

            # 2. Build Segmentation/Mask Contexts
            yolo_labels = []
            for box in meta.get('bbox', []):
                label = box['label']
                if label not in LABEL_MAP:
                    continue
                class_id = LABEL_MAP[label]

                # Extract absolute box points
                xmin, ymin = box['xmin'], box['ymin']
                xmax, ymax = box['xmax'], box['ymax']

                # Since bounding boxes lack custom polygon topologies,
                # we initialize bounding outline masks (4 corner vertices) normalized to [0, 1]
                coords = [
                    (xmin / orig_w), (ymin / orig_h), # Top-Left
                    (xmax / orig_w), (ymin / orig_h), # Top-Right
                    (xmax / orig_w), (ymax / orig_h), # Bottom-Right
                    (xmin / orig_w), (ymax / orig_h)  # Bottom-Left
                ]

                # Flatten to coordinate strings
                coord_str = " ".join([f"{c:.6f}" for c in coords])
                yolo_labels.append(f"{class_id} {coord_str}")

            # Write clean label file mappings
            label_filename = os.path.splitext(filename)[0] + ".txt"
            label_out_path = os.path.join(OUTPUT_DIR, 'labels', split_name, label_filename)
            with open(label_out_path, 'w') as lf:
                lf.write("\n".join(yolo_labels))

    # Execute processing pipelines only if annotations were loaded successfully
    if valid_records:
        process_and_save(train_data, 'train')
        process_and_save(val_data, 'val')
        print("Preprocessing complete! Dataset organized seamlessly under `/content/yolov8_dataset`.")
    else:
        print("Skipping preprocessing as no valid images were found.")

Found 1000 matching files for transformation processing.
Preprocessing complete! Dataset organized seamlessly under `/content/yolov8_dataset`.


In [ ]:
yaml_content = f"""
path: {OUTPUT_DIR}
train: images/train
val: images/val

names:
  0: polyp
"""

with open("/content/dataset.yaml", "w") as f:
    f.write(yaml_content)

# Install Ultralytics and begin instance segmentation training
!pip install ultralytics -q
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")  # Load pre-trained segment model base
model.train(data="/content/dataset.yaml", epochs=25, imgsz=640, device=0)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, 

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d627e28f020>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041, 

In [ ]:
# Install Ultralytics if not already installed
!pip install ultralytics -q

import cv2
import numpy as np
import time
import os
from ultralytics import YOLO
from IPython.display import HTML
from base64 import b64encode

# 1. Load your best trained YOLOv8 model checkpoint
MODEL_PATH = "/content/runs/segment/train/weights/best.pt"  # Corrected path

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model weight file not found at '{MODEL_PATH}'. Please verify training completed successfully.")

model = YOLO(MODEL_PATH)


def analyze_frame_conditions(frame):
    """
    Evaluates frame quality, dye presence, and stool occlusion using image heuristics.
    """
    alerts = []
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Quality Compromised (Blurry or Over/Under-exposed)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    mean_brightness = np.mean(gray)
    if laplacian_var < 50.0 or mean_brightness < 30 or mean_brightness > 220:
        alerts.append("QUALITY COMPROMISED")

    # Dye / Chromoscopy Detected (Indigo carmine / methylene blue hue ranges)
    blue_mask = cv2.inRange(hsv, np.array([90, 50, 50]), np.array([130, 255, 255]))
    blue_ratio = np.sum(blue_mask > 0) / (frame.shape[0] * frame.shape[1])
    if blue_ratio > 0.15:
        alerts.append("DYE DETECTED")

    # Stool / Poor Mucosal View (Yellow/Brownish hue ranges)
    stool_mask = cv2.inRange(hsv, np.array([10, 80, 50]), np.array([30, 255, 255]))
    stool_ratio = np.sum(stool_mask > 0) / (frame.shape[0] * frame.shape[1])
    if stool_ratio > 0.18:
        alerts.append("STOOL CLEARING NEEDED")

    return alerts


def process_custom_video(input_video_path, conf_thresh=0.35):
    """
    Processes a user-specified video file and outputs a flagged version.
    """
    # Clean up whitespace or quotes from copy-pasting
    input_video_path = input_video_path.strip().strip("'").strip('"')

    if not os.path.exists(input_video_path):
        print(f"❌ Error: File not found at path: {input_video_path}")
        return None

    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        print(f"❌ Error opening video stream: {input_video_path}")
        return None

    # Get source video specs
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = int(cap.get(cv2.CAP_PROP_FPS)) or 30
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Output path
    output_video_path = "/content/annotated_output.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    print(f"\n🎬 Processing: {os.path.basename(input_video_path)}")
    print(f"Resolution: {width}x{height} | FPS: {fps} | Total Frames: {total_frames}\n")

    frame_count = 0
    start_time = time.time()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        t0 = time.time()

        # Run inference using the trained model
        results = model.predict(source=frame, conf=conf_thresh, verbose=False)[0]

        annotated_frame = frame.copy()
        polyp_detected = False

        # 1. Annotate Bounding Boxes for Polyps
        if results.boxes is not None and len(results.boxes) > 0:
            polyp_detected = True
            boxes = results.boxes.xyxy.cpu().numpy()
            confs = results.boxes.conf.cpu().numpy()

            for box, conf in zip(boxes, confs):
                x1, y1, x2, y2 = map(int, box)

                # Red Box
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 0, 255), 2)

                # Confidence Tag
                label = f"POLYP {conf:.2f}"
                (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
                cv2.rectangle(annotated_frame, (x1, y1 - 20), (x1 + w, y1), (0, 0, 255), -1)
                cv2.putText(annotated_frame, label, (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

        # Draw Segmentation Polygons if available
        if results.masks is not None:
            masks = results.masks.xy
            for mask in masks:
                pts = np.int32([mask])
                cv2.polylines(annotated_frame, pts, isClosed=True, color=(0, 255, 255), thickness=2)

        # 2. Evaluate Visual Conditions
        frame_alerts = analyze_frame_conditions(frame)
        if polyp_detected:
            frame_alerts.insert(0, "CRITICAL: POLYP DETECTED")

        # 3. Render Status Banner at Top
        if frame_alerts:
            banner_bg = (0, 0, 160) if polyp_detected else (40, 40, 40)
            cv2.rectangle(annotated_frame, (0, 0), (width, 40), banner_bg, -1)

            banner_text = " | ".join(frame_alerts)
            cv2.putText(annotated_frame, banner_text, (15, 26),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)

        # 4. Render FPS counter
        frame_fps = 1.0 / (time.time() - t0 + 1e-6)
        cv2.putText(annotated_frame, f"FPS: {frame_fps:.1f}", (width - 110, height - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)

        out.write(annotated_frame)

        if frame_count % 50 == 0 or frame_count == total_frames:
            print(f"Progress: {frame_count}/{total_frames} frames processed...")

    cap.release()
    out.release()
    elapsed = time.time() - start_time
    print(f"\n✅ Processing Complete in {elapsed:.2f}s! Saved to: {output_video_path}")
    return output_video_path


def play_video_inline(video_path):
    """Encodes and displays video natively in Colab."""
    print("Preparing inline video player...")

    # Re-encode to standard web-compatible h264 format
    web_path = "/content/web_output.mp4"
    os.system(f"ffmpeg -y -i {video_path} -vcodec libx264 -crf 23 {web_path} -loglevel quiet")

    video_file = open(web_path, "rb").read()
    video_url = "data:video/mp4;base64," + b64encode(video_file).decode()
    return HTML(f"""
    <video width="640" height="480" controls autoplay loop>
        <source src="{video_url}" type="video/mp4">
    </video>
    """)


# ==============================================================================
# HOW TO USE: Right-click any video in Colab File Sidebar -> 'Copy path' -> Paste below
# ==============================================================================
user_video_path = input("Enter or paste video file path: ")

output_file = process_custom_video(user_video_path, conf_thresh=0.35)

if output_file:
    display(play_video_inline(output_file))


🎬 Processing: 1_1.avi
Resolution: 768x576 | FPS: 24 | Total Frames: 5001

Progress: 50/5001 frames processed...
Progress: 100/5001 frames processed...
Progress: 150/5001 frames processed...
Progress: 200/5001 frames processed...
Progress: 250/5001 frames processed...
Progress: 300/5001 frames processed...
Progress: 350/5001 frames processed...
Progress: 400/5001 frames processed...
Progress: 450/5001 frames processed...
Progress: 500/5001 frames processed...
Progress: 550/5001 frames processed...
Progress: 600/5001 frames processed...
Progress: 650/5001 frames processed...
Progress: 700/5001 frames processed...
Progress: 750/5001 frames processed...
Progress: 800/5001 frames processed...
Progress: 850/5001 frames processed...
Progress: 900/5001 frames processed...
Progress: 950/5001 frames processed...
Progress: 1000/5001 frames processed...
Progress: 1050/5001 frames processed...
Progress: 1100/5001 frames processed...
Progress: 1150/5001 frames processed...
Progress: 1200/5001 frame